# Experimento — MLP, fase 2: diagnóstico e features

Continua do campeão da primeira execução (`resultados/mlp/mlp-20261103-0337`): MLP padrão com `cliente_novo` e `delta_preco`. **O Optuna e os blocos anteriores não são refeitos.**

1. **Diagnóstico:** ainda existe sinal nas variáveis que o modelo não aproveita?
2. **Blocos de features:** novas formas de `tenure` e das cobranças, regras vindas dos erros e retirada de variáveis.
3. **Arquitetura:** o que o Optuna quase não explorou.

**Regra de aceite:** ganhar em pelo menos 80% das divisões repetidas do treino e superar a régua de ruído, agora medida com 5 sementes. O teste só é usado no fim, com `REVELAR_TESTE = True`.

## Como executar no Kaggle

1. *File → Import Notebook* → `experimentos/mlp/mlp_fase2.ipynb`, com o dataset anexado.
2. *Add-ons → Secrets:* `GITHUB_TOKEN` (com **Contents: Read and write**) e `WANDB_API_KEY`. **Nunca escreva os tokens neste notebook.**
3. *Settings:* Internet **On**.
4. Rode com `MODO_TESTE = True`. Depois, com `False`, use *Save Version → Save & Run All (Commit)*.

# 0. Preparação do ambiente

In [ ]:
# ===== Configuração da execução (edite aqui) =====
REPO_URL = "https://github.com/diegoflyra/churn_project.git"
BRANCH = "diego/mlp-diagnostico"             # branch com o código
RESULTS_REPO_URL = REPO_URL      # onde espelhar os resultados ("" = não enviar ao GitHub)
RESULTS_BRANCH = "resultados"    # branch órfã só de resultados
RUN_NAME = ""                    # vazio = "mlp-fase2-AAAAMMDD-HHMM"; fixe um nome para retomar
RESUME_FROM = ""                 # "" | "github" | pasta com os resultados de uma execução anterior
CSV_PATH = "/kaggle/input/datasets/diegolyra/churn-ibm/telco_ibm.csv"                    # caminho do CSV; vazio = procurar no repositório e em /kaggle/input
FASE_ANTERIOR = "resultados/mlp/mlp-20261103-0337"   # de onde vêm o campeão e o estudo do Optuna (não são refeitos)
N_SEMENTES = 5                   # sementes da avaliação final e das lentes de erro
REPETICOES = 5                   # divisões de treino repetidas no registro evolutivo e nas hipóteses
REVELAR_TESTE = False            # True só com a configuração congelada: o teste é usado uma única vez
USAR_WANDB = True                # False = não envia nada ao W&B, mesmo com a chave
USAR_GITHUB = True               # False = não espelha os resultados no GitHub, mesmo com o token
MODO_TESTE = False               # True = passada rápida para validar o notebook; os números não servem como resultado
HORAS_LIMITE = 10.5              # depois disso não começa trials novos e encerra limpo (o Kaggle corta em 12 h)
BACKUP_MIN = 30                  # envia os resultados ao GitHub a cada N minutos durante a busca

In [ ]:
import importlib.util, os, shutil, subprocess, sys, time
from pathlib import Path

INICIO = time.time()


def _load_dotenv(path):
    """Tokens em um .env local (GITHUB_TOKEN, WANDB_API_KEY); nunca sobrescreve variáveis já definidas."""
    if os.path.isfile(path):
        for line in open(path, encoding="utf-8"):
            key, sep, value = line.strip().partition("=")
            if sep and not key.startswith("#") and value.strip() and not os.environ.get(key.strip()):
                os.environ[key.strip()] = value.strip().strip("'").strip('"')


def _secret(name):
    value = os.environ.get(name, "").strip()
    if value:
        return value
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(name)
    except Exception:
        pass
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        pass
    return ""


def _is_repo(p):
    return (p / "comum.py").exists() and (p / "experimentos" / "mlp" / "modelo.py").exists()


local = next((p for p in [Path.cwd(), *Path.cwd().parents] if _is_repo(p)), None)
if local:
    REPO_DIR = local
    _load_dotenv(REPO_DIR / ".env")
    GITHUB_TOKEN = _secret("GITHUB_TOKEN")
    print(f"Código: cópia local em {REPO_DIR}")
else:
    GITHUB_TOKEN = _secret("GITHUB_TOKEN")
    REPO_DIR = Path("/tmp/churn_project")
    url = REPO_URL.replace("https://", f"https://x-access-token:{GITHUB_TOKEN}@", 1) if GITHUB_TOKEN else REPO_URL
    print("GitHub: usando GITHUB_TOKEN" if GITHUB_TOKEN else "GitHub: clone sem token")
    shutil.rmtree(REPO_DIR, ignore_errors=True)
    if subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", BRANCH, url, str(REPO_DIR)]).returncode != 0:
        raise RuntimeError(f"Falha ao clonar a branch {BRANCH}. Confira se ela foi publicada; repositório privado exige GITHUB_TOKEN.")
    subprocess.run(["git", "-C", str(REPO_DIR), "remote", "set-url", "origin", REPO_URL])  # o token não fica salvo no clone
    print(f"Código: {REPO_DIR}")
os.chdir(REPO_DIR)
print(subprocess.run(["git", "log", "-1", "--oneline"], capture_output=True, text=True).stdout.strip())

# Instala só o que falta: o Kaggle já traz versões compatíveis com a GPU dele
IMPORTS = {"imbalanced-learn": "imblearn", "scikit-learn": "sklearn"}
faltam = [p.strip() for p in open("requirements.txt", encoding="utf-8")
          if p.strip() and importlib.util.find_spec(IMPORTS.get(p.strip(), p.strip())) is None]
if faltam:
    print("Instalando:", ", ".join(faltam))
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *faltam])

In [ ]:
import json

sys.path[:0] = [str(REPO_DIR), str(REPO_DIR / "experimentos" / "mlp")]
import fluxo

WORKDIR = "/kaggle/working" if os.path.isdir("/kaggle/working") else "/content" if os.path.isdir("/content") else str(REPO_DIR)
RUN_NAME = RUN_NAME or time.strftime("mlp-fase2-%Y%m%d-%H%M")
if MODO_TESTE and not RUN_NAME.startswith("teste-"):
    RUN_NAME = "teste-" + RUN_NAME
# O teste nunca se mistura com a execução real, e rodar localmente não sobrescreve o que está versionado em resultados/
PASTA = os.path.join(WORKDIR, "execucoes", RUN_NAME) if WORKDIR == str(REPO_DIR) else os.path.join(WORKDIR, "resultados_" + RUN_NAME)
os.makedirs(PASTA, exist_ok=True)
os.environ["CHURN_RESULTADOS"] = PASTA
os.environ["CHURN_CSV"] = fluxo.achar_csv(CSV_PATH)
if MODO_TESTE:
    os.environ["CHURN_MAX_EPOCAS"] = "5"
    N_SEMENTES, REPETICOES = 2, 2
    print("MODO_TESTE: 5 épocas. Os números não servem como resultado.")

if not USAR_GITHUB:
    RESULTS_REPO_URL = ""
if RESUME_FROM == "github":
    fluxo.baixar_github(PASTA, RUN_NAME, RESULTS_REPO_URL, GITHUB_TOKEN, RESULTS_BRANCH)
elif RESUME_FROM:
    shutil.copytree(RESUME_FROM, PASTA, dirs_exist_ok=True)


def salvar():
    return fluxo.salvar_github(PASTA, RUN_NAME, RESULTS_REPO_URL, GITHUB_TOKEN, RESULTS_BRANCH)


def salvar_json(nome, obj):
    json.dump(obj, open(os.path.join(PASTA, nome), "w"), indent=1)
    salvar()


def tempo_restante():
    return HORAS_LIMITE * 3600 - (time.time() - INICIO)


def continuar():
    return tempo_restante() > 0


import numpy as np, optuna, pandas as pd, torch
from scipy.stats import ks_2samp
import comum, modelo as M, avaliar as A, blocos as BL, diagnostico as D

pd.set_option("display.max_columns", None, "display.width", 200, "display.max_colwidth", 60)
print(f"RUN_NAME = {RUN_NAME} | resultados em {PASTA}")
print(f"CSV: {os.environ['CHURN_CSV']} | device: {M.DEVICE} | GitHub: {'sim' if GITHUB_TOKEN and RESULTS_REPO_URL else 'não'}")

wandb_run = None
wandb_key = _secret("WANDB_API_KEY") if USAR_WANDB else ""
if wandb_key or os.environ.get("WANDB_MODE") == "offline":
    import wandb
    if wandb_key:
        os.environ["WANDB_API_KEY"] = wandb_key
    wandb_run = wandb.init(project="churn-if702", entity=_secret("WANDB_ENTITY") or None, name=RUN_NAME, dir=WORKDIR,
                           group="mlp", config=dict(modelo="mlp", fase=2, lote=M.LOTE, teste=MODO_TESTE))
else:
    print("W&B desativado (sem chave ou USAR_WANDB = False).")


def _apos_bloco(r):
    if wandb_run is not None:
        wandb_run.log({"bloco": r["nome"], "regua": r["regua"], "ks_campeao": max(l["ks"] for l in r["tabela"] if l["decisao"] in ("base", "ACEITA"))})
    salvar()


BL.configurar(PASTA, REPETICOES, _apos_bloco)
if MODO_TESTE:
    BL.N_RUIDO = 1

# Continua da fase anterior: traz o estudo e os blocos, sem treinar de novo
shutil.copytree(os.path.join(REPO_DIR, FASE_ANTERIOR, "blocos"), os.path.join(PASTA, "blocos"), dirs_exist_ok=True)
shutil.copy(os.path.join(REPO_DIR, FASE_ANTERIOR, "mlp.db"), PASTA)
camp = json.load(open(os.path.join(PASTA, "blocos", "b07_preco_por_servico.json")))["campeao"]
print("Campeão herdado:", json.dumps(camp, ensure_ascii=False))


In [ ]:
# Verificações rápidas antes de gastar tempo de treino
folds, final, colunas = comum.carregar_dobras()
ind_val, _ = comum.indices_dobras()
df_ = comum.carregar_df()
for i, f in enumerate(folds):
    assert np.array_equal(df_.loc[ind_val[i], "Churn"].values, f[3].astype(int)), "índices fora de ordem"
    assert abs(f[1].mean() - 0.5) < 1e-6, "treino da dobra deveria estar balanceado"
print(f"{len(df_)} clientes | {len(colunas)} variáveis | validação por dobra: {[len(f[2]) for f in folds]} | índices conferidos")
rng = np.random.default_rng(1); y_ = (rng.random(2000) < .265).astype(int); p_ = np.clip(rng.normal(.3 + .25 * y_, .2), 0, 1)
assert abs(comum.ks(y_, p_) - ks_2samp(p_[y_ == 1], p_[y_ == 0]).statistic) < 1e-9
print("KS conferido contra scipy")

# Diagnóstico do campeão

Só usa previsões de validação. Nada aqui muda o campeão.

- **Sinal restante:** um boosting com as variáveis e o escore do campeão supera o próprio escore? Se não, o sinal disponível já foi usado.
- **Mesmo lado:** entre quem o modelo acusa ou absolve, as variáveis separam os rótulos melhor que o escore?
- **Resíduos:** há grupos ou pares de grupos com churn real diferente do previsto?
- **KS por grupo:** onde o modelo separa pior?
- **Curva de aprendizado:** mais dados ajudariam?

In [ ]:
def diagnosticar(c, rotulo):
    X, y, p = D.oof(c, N_SEMENTES)
    sinal = D.sinal_restante(X, y, p); display(sinal.round(4))
    lado, regras = D.mesmo_lado(X, y, p); display(lado.round(4))
    for k, v in regras.items():
        print(f"Árvore, lado {k}:\n{v}")
    grupos = D.por_grupo(X, y, p); display(grupos.head(10).round(3))
    um, pares = D.residuos(X, y, p)
    print(f"Resíduos com q < 0,05: {int((um.q_bh < .05).sum())} de {len(um)} grupos, {int((pares.q_bh < .05).sum())} de {len(pares)} pares")
    display(pares.head(12).round(3))
    curva = D.curva_aprendizado(c, n_seeds=min(N_SEMENTES, 3)); display(curva.round(4))
    salvar_json(f"mlp_diagnostico_{rotulo}.json", dict(campeao=c, sinal=sinal.to_dict("records"), mesmo_lado=lado.to_dict("records"), regras=regras, por_grupo=grupos.to_dict("records"),
                                                       residuos_grupos=um.to_dict("records"), residuos_pares=pares.head(100).to_dict("records"), curva=curva.to_dict("records")))


if continuar():
    diagnosticar(camp, "inicio")

### 📝 O que o diagnóstico mostra

- **Sobra sinal além do escore?** _…_
- **Algum grupo ou par com erro sistemático?** _…_
- **Mais dados ajudariam?** _…_

# Blocos de features

Cada bloco herda o campeão do anterior. Quando um bloco tem várias variantes, entra só a melhor que passar na regra.

**Para acrescentar um insight:** coloque a variável em `BL.CANDIDATAS`, copie um bloco abaixo e use o próximo número. As regras do `b10` vieram desses mesmos dados, então um resultado positivo nele tende a ser otimista.

## b08_formas_tenure

**Pergunta:** Outras formas de tenure ajudam, além do corte de 6 meses?

In [ ]:
if continuar():
    camp, t = BL.rodar("b08_formas_tenure", "Outras formas de tenure ajudam, além do corte de 6 meses?", {n: dict(extras=[n]) for n in ("tenure_ate_3", "tenure_ate_12", "log_tenure")}, camp)
    display(t.round(4))

## b09_formas_cobranca

**Pergunta:** Colocar as cobranças em escala logarítmica ajuda?

In [ ]:
if continuar():
    camp, t = BL.rodar("b09_formas_cobranca", "Colocar as cobranças em escala logarítmica ajuda?", {n: dict(extras=[n]) for n in ("log_total", "log_mensal")}, camp)
    display(t.round(4))

## b10_regras_erros

**Pergunta:** As regras da árvore sobre clientes do mesmo lado ajudam?

In [ ]:
if continuar():
    camp, t = BL.rodar("b10_regras_erros", "As regras da árvore sobre clientes do mesmo lado ajudam?", {n: dict(extras=[n]) for n in ("novo_sem_fibra_caro", "fibra_recem_chegado", "antigo_sem_automatico")}, camp)
    display(t.round(4))

## b11_retirar_variaveis

**Pergunta:** Retirar variáveis de pouco sinal ou redundantes melhora?

In [ ]:
if continuar():
    camp, t = BL.rodar("b11_retirar_variaveis", "Retirar variáveis de pouco sinal ou redundantes melhora?", {n: dict(extras=[n]) for n in ("sem_gender", "sem_streaming", "sem_total_charges")}, camp)
    display(t.round(4))

## b12_arquitetura

**Pergunta:** Com as features fechadas, outra arquitetura supera o campeão?

In [ ]:
if continuar():
    camp, t = BL.rodar("b12_arquitetura", "Com as features fechadas, outra arquitetura supera o campeão?", {"1 camada, 5": dict(hp=dict(unidades=5)), "1 camada, 32": dict(hp=dict(unidades=32)), "2 camadas, 10": dict(hp=dict(camadas=2)), "2 camadas, 32": dict(hp=dict(camadas=2, unidades=32))}, camp)
    display(t.round(4))

## Evolução do campeão

As duas fases juntas, na ordem dos blocos.

In [ ]:
evo = BL.historico()
display(evo.round(4))
print("Campeão atual:", json.dumps(camp, ensure_ascii=False))
salvar_json("mlp_historico_blocos.json", evo.to_dict("records"))

In [ ]:
# O diagnóstico mudou com o novo campeão?
if continuar():
    diagnosticar(camp, "fim")

### 📝 Análise da fase 2

- **O que foi aceito e o que foi rejeitado?** _…_
- **O diagnóstico mudou?** _…_
- **A arquitetura importa?** _…_

# Avaliação final no teste

Só com a sequência congelada, e uma única vez. `mlp.json` é a linha comparável com os outros modelos, e `mlp_campeao.json` mostra quanto os insights renderam.

In [ ]:
if REVELAR_TESTE and continuar():
    A.main(N_SEMENTES)
    r = json.load(open(os.path.join(PASTA, "mlp.json")))
    fin = BL.final_campeao(camp, N_SEMENTES)
    salvar_json("mlp_campeao.json", fin)
    print(f"Oficial (Optuna): KS = {r['teste']['ks']:.4f} | IC 95% {np.round(r['teste']['ks_ic95'], 4)}")
    print(f"Campeão dos blocos: KS = {fin['teste']['ks']:.4f} | IC 95% {np.round(fin['teste']['ks_ic95'], 4)} | {camp}")
else:
    print("Teste não revelado (REVELAR_TESTE = False).")

In [ ]:
salvar()
if wandb_run is not None:
    wandb_run.summary.update(dict(campeao=json.dumps(camp)))
    wandb_run.finish()
print("Resultados em", PASTA)
print(sorted(os.listdir(PASTA)))